# مشروع مقرر الشبكات العصبية المتكررة (RNN)
## تحليل مشاعر مراجعات الأفلام باستخدام LSTM على مجموعة بيانات IMDB
- **الكلية التطبيقية - الجامعة السعودية الإلكترونية**
- **أعضاء المجموعة:**
  1. [اسم الطالب 1] - [الرقم الجامعي] (تحميل واستكشاف البيانات وتجهيزها)
  2. [اسم الطالب 2] - [الرقم الجامعي] (بناء وتدريب نموذج LSTM)
  3. [اسم الطالب 3] - [الرقم الجامعي] (رسم المنحنيات وحساب مؤشرات التقييم)
  4. [اسم الطالب 4] - [الرقم الجامعي] (مصفوفة الالتباس وتحليل الأخطاء)
  5. [اسم الطالب 5] - [الرقم الجامعي] (توثيق الكود وإعداد التقرير النهائي)

### 1. استيراد المكتبات وتحميل مجموعة بيانات IMDB واستكشافها

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# تثبيت البذرة العشوائية لضمان اتساق النتائج
tf.random.set_seed(42)
np.random.seed(42)

# تحديد حجم المفردات (أكثر 10,000 كلمة تكراراً)
vocab_size = 10000

# تحميل البيانات من Keras (25,000 للتدريب و25,000 للاختبار)
(X_train_full, y_train_full), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print("عدد عينات التدريب الكلية:", len(X_train_full))
print("عدد عينات الاختبار:", len(X_test))
print("الفئات الموجودة:", np.unique(y_train_full), "(0 = سلبي، 1 = إيجابي)")
print("توزيع الفئات في التدريب:", np.bincount(y_train_full))
print("توزيع الفئات في الاختبار:", np.bincount(y_test))

# استكشاف أطوال المراجعات لتبرير اختيار طول التسلسل الموحد
lengths = [len(seq) for seq in X_train_full]
print(f"متوسط الطول: {np.mean(lengths):.1f} | الوسيط: {np.median(lengths):.1f} | الحد الأقصى: {np.max(lengths)}")
print("أول مراجعة كتسلسل رقمي (أول 15 عنصراً):", X_train_full[0][:15])

### 2. توحيد طول التسلسلات (Padding / Truncating) واقتطاع مجموعة التحقق (Validation)

In [ ]:
maxlen = 200

# توحيد طول التسلسلات عند 200 كلمة للموازنة بين السياق وسرعة التدريب
X_train_full_pad = pad_sequences(X_train_full, maxlen=maxlen, padding="post", truncating="post")
X_test_pad       = pad_sequences(X_test,       maxlen=maxlen, padding="post", truncating="post")

# اقتطاع 20% من بيانات التدريب لمجموعة التحقق (Validation) مع إبقاء الاختبار للتقييم النهائي فقط
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full_pad, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

print("شكل بيانات التدريب (Train):", X_train.shape, y_train.shape)
print("شكل بيانات التحقق (Validation):", X_val.shape, y_val.shape)
print("شكل بيانات الاختبار (Test):", X_test_pad.shape, y_test.shape)

### 3. بناء نموذج LSTM وتدريبه

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=maxlen),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=6,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

### 4. رسم منحنيات الدقة والخسارة وحساب مؤشرات الأداء على بيانات الاختبار

In [ ]:
# رسم منحنيات Accuracy و Loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], marker="o", label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], marker="s", label="Validation Accuracy")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["loss"], marker="o", label="Train Loss")
axes[1].plot(history.history["val_loss"], marker="s", label="Validation Loss")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# التقييم النهائي على بيانات الاختبار فقط
y_prob = model.predict(X_test_pad, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [
        round(accuracy_score(y_test, y_pred), 4),
        round(precision_score(y_test, y_pred), 4),
        round(recall_score(y_test, y_pred), 4),
        round(f1_score(y_test, y_pred), 4)
    ]
})

print("\nمؤشرات الأداء على بيانات الاختبار:")
print(metrics_df.to_string(index=False))

### 5. مصفوفة الالتباس وعرض أمثلة للمراجعات المصنفة خطأً

In [ ]:
# مصفوفة الالتباس (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative (0)", "Positive (1)"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - IMDB LSTM")
plt.show()

# إعادة تحويل الأرقام إلى كلمات باستخدام get_word_index()
word_index = imdb.get_word_index()
index_to_word = {idx + 3: word for word, idx in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<OOV>"
index_to_word[3] = "<UNK>"

def decode_review(seq):
    return " ".join(index_to_word.get(i, "?") for i in seq if i != 0)

errors_idx = np.where(y_test != y_pred)[0]
print(f"\nعدد المراجعات المصنفة خطأً في الاختبار: {len(errors_idx)} من أصل {len(y_test)}")

label_names = {0: "Negative", 1: "Positive"}

for idx in errors_idx[:5]:
    decoded = decode_review(X_test[idx])
    print("=" * 75)
    print(f"العينة #{idx} | الفعلي: {label_names[y_test[idx]]} | المتوقع: {label_names[y_pred[idx]]} | الاحتمال: {y_prob[idx]:.3f}")
    print("النص (أول 55 كلمة):", " ".join(decoded.split()[:55]))